In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, explained_variance_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PowerTransformer, StandardScaler
from gplearn.genetic import SymbolicRegressor
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

# Metrics + Plots
def metrics(y_true, y_pred, n_features=None):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Adj_R2": 1 - (1 - r2_score(y_true, y_pred)) * (len(y_true) - 1) / (len(y_true) - (n_features if n_features else 1) - 1),
        "ExplainedVar": explained_variance_score(y_true, y_pred)
    }

def plot_residuals(y_true, y_pred, title):
    plt.figure()
    residuals = y_true - y_pred
    plt.scatter(y_pred, residuals, alpha=0.6, edgecolor="k")
    plt.axhline(y=0, color="red", linestyle="--")
    plt.title(f"{title} Residuals")
    plt.xlabel("Predicted")
    plt.ylabel("Residuals")
    plt.show()

def plot_pred_vs_actual(y_true, y_pred, title):
    plt.figure()
    plt.scatter(y_true, y_pred, alpha=0.6, edgecolor="k")
    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], "r--")
    plt.title(f"{title}: Predicted vs Actual")
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.show()

# Model Eval
def evaluate_model_full(name, model, X, y, transform=None, random_state=42):
    results = {}

    if transform == "sqrt":
        y_trans = np.sqrt(y)
        inv = lambda yp: yp ** 2
    elif transform == "log1p":
        y_trans = np.log1p(y)
        inv = lambda yp: np.expm1(yp)
    elif transform in ["yeo-johnson", "box-cox"]:
        pt = PowerTransformer(method=transform)
        y_shift = y + 1e-3 if transform == "box-cox" else y
        y_trans = pt.fit_transform(y_shift.reshape(-1, 1)).ravel()
        inv = lambda yp: pt.inverse_transform(yp.reshape(-1, 1)).ravel()
    else:
        y_trans = y
        inv = lambda yp: yp

    #In-Samp
    model.fit(X, y_trans)
    y_pred = inv(model.predict(X))
    results["In-Sample"] = metrics(y, y_pred, n_features=X.shape[1])
    print(f"\n{name} In-Sample Metrics:\n", results["In-Sample"])
    plot_residuals(y, y_pred, f"{name} (In-Sample)")
    plot_pred_vs_actual(y, y_pred, f"{name} (In-Sample)")

    #Validation (80/20)
    X_train, X_test, y_train, y_test = train_test_split(X, y_trans, test_size=0.2, random_state=random_state)
    model.fit(X_train, y_train)
    y_pred = inv(model.predict(X_test))
    results["Validation"] = metrics(inv(y_test), y_pred, n_features=X.shape[1])
    print(f"\n{name} Validation Metrics:\n", results["Validation"])
    plot_residuals(y[ len(y_train): ], y_pred, f"{name} (Validation)")
    plot_pred_vs_actual(y[ len(y_train): ], y_pred, f"{name} (Validation)")

    #5x Cross
    kf = KFold(n_splits=5, shuffle=True, random_state=random_state)
    cv_metrics = []
    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y_trans[train_idx], y[test_idx]
        model.fit(X_train, y_train)
        y_pred = inv(model.predict(X_test))
        cv_metrics.append(metrics(y_test, y_pred, n_features=X.shape[1]))

    results["CrossVal_5x"] = {k: np.mean([m[k] for m in cv_metrics]) for k in cv_metrics[0]}
    print(f"\n{name} 5x Cross-Validation Metrics:\n", results["CrossVal_5x"])

    return results

def run_feature_selection(X, y, method="forward", n_features="best"):
    lr = LinearRegression()
    if method == "forward":
        sfs = SFS(lr, k_features=n_features, forward=True, floating=False, scoring='r2', cv=5)
    elif method == "backward":
        sfs = SFS(lr, k_features=n_features, forward=False, floating=False, scoring='r2', cv=5)
    elif method == "stepwise":
        sfs = SFS(lr, k_features=n_features, forward=True, floating=True, scoring='r2', cv=5)
    else:
        raise ValueError("method must be 'forward', 'backward', or 'stepwise'")

    sfs.fit(X, y)
    return list(sfs.k_feature_idx_)

RANDOM_STATE = 42
df = pd.read_csv("../data/appliances-energy.csv")
df = df.drop(columns=["date"])
y = df["Appliances"].values
X = df.drop(columns=["Appliances"])
feature_names = X.columns.tolist()
X = StandardScaler().fit_transform(X)
print("\n- Descriptive Statistics -")
print(df.describe())

sns.pairplot(df.iloc[:, :6], diag_kind="kde")
plt.suptitle("Pairplot of Appliances Energy Dataset", y=1.02)
plt.show()
plt.figure(figsize=(12,10))
sns.heatmap(df.corr(), annot=False, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

# Models
base_models = {
    "Linear": (LinearRegression(), None),
    "Ridge": (Ridge(alpha=1.0), None),
    "Lasso": (Lasso(alpha=0.01, max_iter=10000), None),
    "Bridge": (ElasticNet(alpha=0.05, l1_ratio=0.5), None),
    "Sqrt": (LinearRegression(), "sqrt"),
    #"Log1p": (LinearRegression(), "log1p"),
    #"Box-Cox": (LinearRegression(), "box-cox"),
    #"Yeo-Johnson": (LinearRegression(), "yeo-johnson"),
    #"Symbolic": (SymbolicRegressor(population_size=1000, generations=10,
    #                               function_set=["add", "sub", "mul", "div"],
    #                               metric="rmse", random_state=RANDOM_STATE, verbose=1), None)
}

all_results = {}

for name, (model, transform) in base_models.items():
    all_results[f"{name} (all)"] = evaluate_model_full(f"{name} (All Features)", model, X, y, transform=transform)
    for fs_method in ["forward", "backward", "stepwise"]:
        idx = run_feature_selection(X, y, method=fs_method, n_features=5)
        X_fs = X[:, idx]
        all_results[f"{name} ({fs_method})"] = evaluate_model_full(f"{name} ({fs_method} FS)", model, X_fs, y, transform=transform)

# Final Result
print("\n\n- Summary of Models (with FS) -\n")
for model, res in all_results.items():
    print(f"\n- {model} -")
    for k, v in res.items():
        print(f"{k}: {v}")

FileNotFoundError: [Errno 2] No such file or directory: 'energydata_complete.csv'